# CSCE 676 — Semester Project (Final Deliverable)
## Mining Frequent, Associative, and Sequential Patterns in MovieLens 25M

**Dataset:** MovieLens 25M (GroupLens / University of Minnesota) — 25 million ratings from 162,541 users across 62,423 movies.
**Course techniques used:** Frequent itemset mining, association rule mining.
**External technique used:** Sequential pattern mining (PrefixSpan).

### Headline Results

This notebook answers three research questions using MovieLens 25M at full scale (25,000-user × 2,500-item basket matrix, plus a 6,000-user sequential sample).

- **RQ1 (confirmed).** Frequent itemsets grow from 563 at support=0.10 to 410,690 at support=0.015, mostly driven by length-3 combinations. Changing the support threshold changes *what kind of content* the model finds, not just how much.
- **RQ2 (confirmed, strongly).** Ranking rules by confidence vs. lift gives completely different results: Jaccard overlap = 0.00 at K=10, 25, 50, and 100. Confidence favors rules pointing at popular movies (Godfather); lift favors franchise pairs (Harry Potter, Bourne). The two metrics are orthogonal.
- **RQ3 (disconfirmed).** Across three PrefixSpan min_support levels, all 4,412 sequential patterns (including all 383 of length ≥ 3) had a frequent unordered counterpart. Sequence ordering adds no new information beyond what unordered mining already captures.

Each RQ has its own section with the question, plan, code, tables/plots, and findings. Sections 9–11 cover synthesis, limitations, and conclusions.

### Runtime Environment

The earlier checkpoints ran on Google Colab, but the full-scale analysis needed more than Colab's 12 GB RAM limit. This final version runs locally on a Windows workstation and loads data from a local directory (§4). All results come from a single top-to-bottom run.

Requirements: Python 3.10+, `pandas`, `numpy`, `scipy`, `matplotlib`, `mlxtend`, `prefixspan`. Place `ratings.csv` and `movies.csv` in the directory set in §4. Full runtime is about 25–30 minutes on a mid-range desktop.


## 1. Motivation

From the Checkpoint 2 EDA, MovieLens has some important characteristics that shape how pattern mining works on it:

- A small number of movies get most of the ratings (skewed item popularity).
- A small number of users produce most of the ratings (skewed user activity).
- Positive basket sizes are large and long-tailed (median 40, 95th percentile 261, max 5,525).
- Rating activity is temporally clustered, not spread out evenly over time.

These observations lead to three hypotheses:

1. Frequent itemset counts should change a lot as we lower the support threshold — high supports will only show blockbusters, while low supports should reveal niche content.
2. Confidence and lift should rank rules very differently, because confidence rewards popularity while lift corrects for it.
3. Sequential pattern mining might find structure that unordered mining misses — patterns where the *order* matters, not just which items appear together.

The rest of the notebook tests each hypothesis across multiple parameter settings so the results are not tied to a single threshold choice.


## 2. Research Questions

**RQ1.** What frequent itemsets emerge under different minimum support thresholds when transactions are defined by positively rated movies (rating ≥ 4.0)?

**RQ2.** How do confidence and lift differ when ranking association rules from positive user baskets?

**RQ3.** Do sequential patterns in user rating histories reveal structure that unordered frequent itemsets miss?

| RQ  | Task Type                   | Course / External | Algorithm(s)            | Evaluation Criteria                                              |
|-----|-----------------------------|-------------------|--------------------------|------------------------------------------------------------------|
| RQ1 | Frequent itemset mining     | Course            | Apriori + FP-Growth     | support, # itemsets, itemset length, diversity, interpretability |
| RQ2 | Association rule mining     | Course            | Apriori + rule generation| confidence, lift, support, redundancy, overlap                   |
| RQ3 | Sequential pattern mining   | External          | PrefixSpan              | sequential support, novelty vs unordered, interpretability       |


## 3. Environment Setup

Install dependencies in the local Python environment before running this notebook:

```
pip install pandas numpy scipy matplotlib mlxtend prefixspan jupyter
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import random
import time
import gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from prefixspan import PrefixSpan

plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 4. Data Loading (Local)

Set `DATA_DIR` to point at the folder containing `ratings.csv` and `movies.csv` from the MovieLens 25M download. On Windows, use a raw string like `DATA_DIR = Path(r"C:\Users\you\datasets\ml-25m")`.


In [ ]:
# Edit this path to point at your dataset folder.
DATA_DIR = Path("ml-25m")

if not (DATA_DIR / "ratings.csv").exists():
    raise FileNotFoundError(
        f"ratings.csv not found in {DATA_DIR.resolve()}.\n"
        "Download MovieLens 25M from https://grouplens.org/datasets/movielens/25m/\n"
        "and unzip so that ratings.csv and movies.csv live inside DATA_DIR."
    )

ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(DATA_DIR / "movies.csv")

print("ratings shape:", ratings.shape, "  memory:", f"{ratings.memory_usage(deep=True).sum()/1e6:.0f} MB")
print("movies shape:", movies.shape)

title_of = movies.set_index("movieId")["title"].to_dict()

def titles(ids):
    return [title_of.get(int(i), f"movieId={i}") for i in ids]


## 5. Preprocessing: Positive Baskets

A movie counts as part of a user's transaction if they rated it **4.0 or higher**. This keeps only positive preferences, which is what makes itemset and rule mining meaningful on rating data.


In [ ]:
POS_RATING_THRESHOLD = 4.0

positive_ratings = ratings.loc[
    ratings["rating"] >= POS_RATING_THRESHOLD,
    ["userId", "movieId", "timestamp"]
].copy()

n_tx = positive_ratings["userId"].nunique()
n_items = positive_ratings["movieId"].nunique()

print(f"Positive-rating events : {len(positive_ratings):,}")
print(f"Transactions (users)   : {n_tx:,}")
print(f"Unique items (movies)  : {n_items:,}")
print(f"positive_ratings memory: {positive_ratings.memory_usage(deep=True).sum()/1e6:.0f} MB")


### 5.1 Pruning the Item Vocabulary

We keep only the **top 2,500 most-popular movies** (ranked by how many users rated them positively). This is large enough to capture long-tail patterns at low supports while keeping the basket matrix manageable in memory.


In [ ]:
TOP_K_ITEMS = 2500

item_user_counts = positive_ratings.groupby("movieId")["userId"].nunique()
kept_items = item_user_counts.sort_values(ascending=False).head(TOP_K_ITEMS).index
kept_items_set = set(kept_items.tolist())

pruned = positive_ratings[positive_ratings["movieId"].isin(kept_items_set)].copy()

print(f"Kept items (top {TOP_K_ITEMS} by positive raters): {len(kept_items):,}")
print(f"Rows remaining: {len(pruned):,}  ({len(pruned)/len(positive_ratings):.1%} of positives)")
print(f"pruned memory: {pruned.memory_usage(deep=True).sum()/1e6:.0f} MB")


### 5.2 Building the Transaction Matrix

We sample 25,000 users and build a dense boolean basket matrix (25k × 2.5k = ~60 MB). Dense boolean format is the fastest input for mlxtend's mining functions.


In [ ]:
USER_SAMPLE_SIZE = 25000

all_users = pruned["userId"].unique()
sample_users = np.random.default_rng(RANDOM_SEED).choice(
    all_users, size=min(USER_SAMPLE_SIZE, len(all_users)), replace=False
)

sample = pruned[pruned["userId"].isin(sample_users)][["userId", "movieId"]].drop_duplicates()

user_codes, user_index = pd.factorize(sample["userId"], sort=False)
item_codes, item_index = pd.factorize(sample["movieId"], sort=False)

n_users = len(user_index)
n_movies = len(item_index)

csr = csr_matrix(
    (np.ones(len(sample), dtype=bool), (user_codes, item_codes)),
    shape=(n_users, n_movies),
)

dense = csr.toarray()
basket_df = pd.DataFrame(
    dense,
    columns=[str(x) for x in item_index.tolist()],
    copy=False,
)
del dense

print(f"Sampled users: {n_users:,}")
print(f"Items in vocab: {n_movies:,}")
print(f"Basket matrix shape: {basket_df.shape}")
print(f"Density: {csr.nnz / (n_users * n_movies):.4%}")
print(f"Dense memory: {basket_df.memory_usage(deep=True).sum()/1e6:.1f} MB")


## 6. RQ1 — Frequent Itemsets Under Varying Minimum Support

**Question:** What frequent itemsets emerge under different minimum support thresholds?

**Plan:** Sweep support across six levels from 0.10 down to 0.015, with `max_len=3` at every level so we can see length-3 patterns even at the lowest thresholds.


In [ ]:
# FP-Growth is the primary miner (low memory). Apriori is used as a
# correctness check at high supports only (it OOMs at low supports on this matrix).
SUPPORTS = [0.10, 0.07, 0.05, 0.03, 0.02, 0.015]
MAX_LEN_BY_SUPPORT = {s: 3 for s in SUPPORTS}
APRIORI_CHECK_SUPPORTS = [0.10, 0.07]

rq1_rows = []
itemsets_by_support = {}

sweep_t0 = time.time()
print(f"[start] RQ1 sweep over {len(SUPPORTS)} support levels")
print(f"        basket_df shape: {basket_df.shape}")
print("-" * 70)

for i, s in enumerate(SUPPORTS, 1):
    ml = MAX_LEN_BY_SUPPORT[s]
    print(f"[{i}/{len(SUPPORTS)}] min_support = {s:.3f}  (max_len={ml})")

    print(f"    fpgrowth(min_support={s:.3f}) ... ", end="", flush=True)
    t0 = time.time()
    fp = fpgrowth(basket_df, min_support=s, use_colnames=True, max_len=ml)
    t_fp = time.time() - t0
    print(f"done in {t_fp:7.2f}s  -> {len(fp):>8,} itemsets")

    fp = fp.copy()
    fp["length"] = fp["itemsets"].apply(len)
    itemsets_by_support[s] = fp

    row = {
        "min_support": s,
        "max_len_cap": ml,
        "n_itemsets_fpgrowth": len(fp),
        "avg_length": fp["length"].mean() if len(fp) else 0,
        "max_length": int(fp["length"].max()) if len(fp) else 0,
        "fpgrowth_sec": round(t_fp, 2),
    }

    # Apriori cross-check at high supports only (low_memory=True to avoid OOM).
    if s in APRIORI_CHECK_SUPPORTS:
        print(f"    apriori(min_support={s:.3f}, low_memory=True) ... ", end="", flush=True)
        t0 = time.time()
        ap = apriori(basket_df, min_support=s, use_colnames=True,
                     max_len=ml, low_memory=True)
        t_ap = time.time() - t0
        print(f"done in {t_ap:7.2f}s  -> {len(ap):>8,} itemsets")
        row["n_itemsets_apriori"] = len(ap)
        row["apriori_sec"] = round(t_ap, 2)
        if len(ap) != len(fp):
            print(f"    NOTE: apriori and fpgrowth differ by {abs(len(ap)-len(fp))} "
                  f"(boundary ties)")
        else:
            print(f"    correctness check OK (apriori == fpgrowth)")
        del ap
        gc.collect()
    else:
        row["n_itemsets_apriori"] = None
        row["apriori_sec"] = None

    elapsed = time.time() - sweep_t0
    print(f"    cumulative elapsed: {elapsed:7.1f}s")
    print("-" * 70)

    rq1_rows.append(row)

print(f"[done] RQ1 sweep finished in {time.time()-sweep_t0:.1f}s total")

rq1_df = pd.DataFrame(rq1_rows)[[
    "min_support", "max_len_cap", "n_itemsets_fpgrowth", "n_itemsets_apriori",
    "avg_length", "max_length", "fpgrowth_sec", "apriori_sec"
]]
rq1_df


### 6.1 How does itemset size change with support?


In [ ]:
size_dist = []
for s, df in itemsets_by_support.items():
    if len(df) == 0:
        continue
    counts = df["length"].value_counts().sort_index()
    for k, v in counts.items():
        size_dist.append({"min_support": s, "itemset_length": k, "count": int(v)})

size_df = pd.DataFrame(size_dist).pivot(
    index="min_support", columns="itemset_length", values="count"
).fillna(0).astype(int)
size_df.columns = [f"len={c}" for c in size_df.columns]
size_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col in size_df.columns:
    ser = size_df[col]
    ax.plot(ser.index, ser.values, marker="o", label=col)
ax.set_xscale("log")
ax.set_yscale("log")
ax.invert_xaxis()
ax.set_xlabel("Minimum support (log scale; stricter on the left)")
ax.set_ylabel("# frequent itemsets (log scale)")
ax.set_title("RQ1: Frequent itemset count vs. support, by itemset length")
ax.legend()
plt.tight_layout()
plt.show()


### 6.2 What Do the Actual Itemsets Look Like?

Simply showing "top 10 by support" at each threshold repeats the same popular movies every time. Instead, we use a **band view**: for each threshold, we show patterns whose support falls *between* that threshold and the next-stricter one. This highlights what new patterns appeared because we lowered the threshold.


In [ ]:
def show_top_itemsets(support_level, length, k=10):
    df = itemsets_by_support[support_level]
    df = df[df["length"] == length].sort_values("support", ascending=False).head(k).copy()
    df["titles"] = df["itemsets"].apply(lambda s: titles(list(s)))
    return df[["support", "length", "titles"]].reset_index(drop=True)

def show_band_itemsets(lower_support, upper_support, length, k=10, source_support=None):
    """Top-k itemsets with support in [lower, upper)."""
    src = source_support if source_support is not None else lower_support
    df = itemsets_by_support[src]
    df = df[(df["length"] == length) &
            (df["support"] >= lower_support) &
            (df["support"] <  upper_support)].sort_values("support", ascending=False).head(k).copy()
    df["titles"] = df["itemsets"].apply(lambda s: titles(list(s)))
    return df[["support", "length", "titles"]].reset_index(drop=True)

# Top itemsets at the strictest support (blockbusters).
print("=== Top singletons at support = 0.10 ===")
display(show_top_itemsets(0.10, length=1))

print("\n=== Top pairs at support = 0.10 ===")
display(show_top_itemsets(0.10, length=2))

# Band views: patterns unique to each support range.
print("\n=== Pairs in band [0.05, 0.07) ===")
display(show_band_itemsets(0.05, 0.07, length=2, source_support=0.05))

print("\n=== Triples in band [0.05, 0.07) ===")
display(show_band_itemsets(0.05, 0.07, length=3, source_support=0.05))

print("\n=== Pairs in band [0.03, 0.05) ===")
display(show_band_itemsets(0.03, 0.05, length=2, source_support=0.03))

print("\n=== Triples in band [0.03, 0.05) ===")
display(show_band_itemsets(0.03, 0.05, length=3, source_support=0.03))

print("\n=== Pairs in band [0.015, 0.02) ===")
display(show_band_itemsets(0.015, 0.02, length=2, source_support=0.015))

print("\n=== Triples in band [0.015, 0.02) ===")
display(show_band_itemsets(0.015, 0.02, length=3, source_support=0.015))


### 6.3 RQ1 Findings

The sweep confirms the RQ1 hypothesis:

- **Itemset counts grow dramatically as support drops.** Counts go from 563 → 2,454 → 9,175 → 53,940 → 185,295 → 410,690 across the six support levels (0.10 → 0.015). Most of that growth comes from length-3 combinations: at support=0.015 there are 370,362 triples vs. 39,177 pairs.
- **Average itemset length increases too.** It climbs from 2.07 at support=0.10 to 2.90 at support=0.015. At strict supports, length-3 patterns are nearly invisible (145 triples at 0.10), but they become the dominant category as support drops (43,706 triples at 0.03).
- **The band view reveals how content changes at each threshold.** At high supports (0.07–0.10), we see blockbuster co-watches like Seven↔Star Wars. In the 0.05–0.07 band, genre-cohort pairs and franchise triples start appearing. At 0.03–0.05, auteur and cross-franchise co-watches show up. Below 0.02, we reach the long tail — niche cult-film triples that are real patterns but too rare to clear a 3% threshold.
- **FP-Growth vs. Apriori.** On this 25,000 × 2,500 matrix, mlxtend's default Apriori runs out of memory at low supports because it builds large boolean tensors internally. We used FP-Growth (which uses a prefix-tree and stays low on RAM) as the primary miner, and cross-checked with Apriori (using `low_memory=True`) at supports 0.10 and 0.07. Both algorithms agreed exactly at those levels (563 and 2,454 itemsets), confirming the results.

**RQ1 is confirmed.** Changing the support threshold is not just a tuning decision — it determines what kind of content the model sees. The band view makes this concrete with specific examples at every level.


## 7. RQ2 — Confidence vs. Lift for Association Rules

**Question:** How do confidence and lift differ when ranking association rules from positive user baskets?

**Plan:** Generate all rules at support=0.05, rank them by confidence and by lift separately, and measure how much the two rankings overlap using Jaccard similarity at K=10, 25, 50, and 100.


In [ ]:
RULE_SUPPORT = 0.05
MIN_CONFIDENCE = 0.5

freq = itemsets_by_support[RULE_SUPPORT]
rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
print(f"# rules at support={RULE_SUPPORT}, min_confidence={MIN_CONFIDENCE}: {len(rules):,}")

rules["antecedent_titles"] = rules["antecedents"].apply(lambda s: titles(list(s)))
rules["consequent_titles"] = rules["consequents"].apply(lambda s: titles(list(s)))
rules = rules[["antecedent_titles", "consequent_titles",
               "support", "confidence", "lift", "antecedent support", "consequent support"]]
rules.head()


### 7.1 Top-10 by confidence vs. top-10 by lift


In [ ]:
top_conf = rules.sort_values("confidence", ascending=False).head(10).reset_index(drop=True)
top_lift = rules.sort_values("lift",       ascending=False).head(10).reset_index(drop=True)

print("=== Top 10 rules by CONFIDENCE ===")
display(top_conf)

print("\n=== Top 10 rules by LIFT ===")
display(top_lift)


In [ ]:
# Measure overlap between confidence-ranked and lift-ranked top-K.
def rule_key(row):
    return (tuple(sorted(row["antecedent_titles"])), tuple(sorted(row["consequent_titles"])))

for K in [10, 25, 50, 100]:
    top_c = set(rules.nlargest(K, "confidence").apply(rule_key, axis=1))
    top_l = set(rules.nlargest(K, "lift").apply(rule_key, axis=1))
    overlap = len(top_c & top_l)
    jacc = overlap / len(top_c | top_l) if (top_c | top_l) else 0.0
    print(f"Top-{K:>3}: overlap={overlap:>3}, Jaccard={jacc:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(rules["confidence"], rules["lift"],
                c=rules["consequent support"], cmap="viridis", s=14, alpha=0.7)
ax.set_xlabel("confidence")
ax.set_ylabel("lift")
ax.set_title("RQ2: confidence vs. lift, colored by consequent popularity")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("consequent support (popularity)")
plt.tight_layout()
plt.show()


### 7.2 RQ2 Findings

At support=0.05 and min_confidence=0.5, we generated **14,442 rules**.

- **Confidence-top rules point at popular movies.** All ten top-confidence rules have *The Godfather* as the consequent. The highest is `[Godfather Part II, Apocalypse Now] → Godfather` at confidence=0.966. Confidence is essentially tracking how popular the consequent movie is, since confidence = P(consequent | antecedent) gets inflated for widely-liked movies.
- **Lift-top rules are franchise pairs.** The highest-lift rule is `[HP Goblet of Fire] → [HP Prisoner of Azkaban]` at lift=9.29, followed by more Harry Potter and Bourne pairs. These are movies that are watched together more than their individual popularity would predict — exactly what lift is designed to find.
- **The two rankings share zero rules.** Jaccard overlap is exactly 0.000 at K=10, 25, 50, and 100. This means a recommender using only confidence would never surface the rules that lift promotes, and vice versa. The disagreement is not marginal — it is complete.
- **The scatter plot shows this visually.** High-confidence rules cluster where consequent popularity is high (yellow dots), while high-lift rules sit where consequent popularity is low (purple dots). The two metrics pull in opposite directions.

**RQ2 is confirmed, strongly.** Confidence and lift do not just disagree — they select completely disjoint rule sets, which is a stronger result than expected.


## 8. RQ3 — Sequential Patterns via PrefixSpan

**Question:** Do sequential patterns in user rating histories reveal structure that unordered frequent itemsets miss?

**Plan:** Run PrefixSpan at three min_support levels (0.10, 0.05, 0.03) on a 6,000-user timestamp-sorted sample. For each level, compare the sequential patterns against the unordered itemsets from RQ1 at the same support and count how many are truly novel (no unordered counterpart). Using three levels ensures the result is not tied to a single threshold choice.


In [ ]:
# Build per-user timestamp-sorted sequences from the pruned item set.
SEQ_MIN_LEN = 10
SEQ_MAX_LEN = 300
SEQ_USER_SAMPLE = 6000

seq_source = pruned.sort_values(["userId", "timestamp"])
user_seqs_all = seq_source.groupby("userId")["movieId"].apply(list)

user_seqs_filtered = user_seqs_all[
    (user_seqs_all.str.len() >= SEQ_MIN_LEN) &
    (user_seqs_all.str.len() <= SEQ_MAX_LEN)
]

rng = np.random.default_rng(RANDOM_SEED)
seq_sample_users = rng.choice(user_seqs_filtered.index.values,
                              size=min(SEQ_USER_SAMPLE, len(user_seqs_filtered)),
                              replace=False)
sequences = [list(user_seqs_filtered.loc[u]) for u in seq_sample_users]

del seq_source, user_seqs_all, user_seqs_filtered
gc.collect()

print(f"# sequences for PrefixSpan: {len(sequences):,}")
print(f"  mean length: {np.mean([len(s) for s in sequences]):.1f}")
print(f"  median length: {int(np.median([len(s) for s in sequences]))}")


In [ ]:
# Run PrefixSpan at three min_support levels and compare against RQ1 unordered itemsets.
PREFIXSPAN_MINSUP_FRACS = [0.10, 0.05, 0.03]
PREFIXSPAN_MAXLEN = 5

rq3_sensitivity = []   # one row per (minsup_frac) setting
rq3_patterns = {}      # minsup_frac -> DataFrame of patterns at that setting

for minsup_frac in PREFIXSPAN_MINSUP_FRACS:
    t0 = time.time()
    minsup_abs = max(2, int(minsup_frac * len(sequences)))

    ps = PrefixSpan(sequences)
    ps.minlen = 2
    ps.maxlen = PREFIXSPAN_MAXLEN

    patterns = ps.frequent(minsup_abs, closed=True)
    t_ps = time.time() - t0

    df = pd.DataFrame([{
        "seq_support": cnt,
        "seq_support_frac": cnt / len(sequences),
        "pattern_movieIds": pat,
        "pattern_titles": titles(pat),
        "length": len(pat),
    } for cnt, pat in patterns])

    # Get matching unordered itemsets from RQ1 for comparison.
    if minsup_frac in itemsets_by_support:
        ref_support = minsup_frac
    else:
        ref_support = min(itemsets_by_support.keys(),
                          key=lambda x: (abs(x - minsup_frac), x))
    ref_itemsets = set(
        itemsets_by_support[ref_support]["itemsets"].apply(
            lambda s: frozenset(int(x) for x in s)
        )
    )

    if len(df):
        df["as_set"] = df["pattern_movieIds"].apply(lambda ids: frozenset(int(x) for x in ids))
        df["unordered_counterpart_frequent"] = df["as_set"].isin(ref_itemsets)
        df["novel_vs_unordered"] = ~df["unordered_counterpart_frequent"]
        n_novel = int(df["novel_vs_unordered"].sum())
    else:
        n_novel = 0

    rq3_patterns[minsup_frac] = df
    rq3_sensitivity.append({
        "minsup_frac": minsup_frac,
        "ref_support_in_RQ1": ref_support,
        "n_patterns": len(df),
        "n_novel": n_novel,
        "novel_frac": (n_novel / len(df)) if len(df) else 0.0,
        "prefixspan_sec": round(t_ps, 2),
    })

    print(f"minsup={minsup_frac} (abs {minsup_abs}): "
          f"{len(df):>5} patterns, {n_novel:>3} novel, "
          f"runtime {t_ps:6.1f}s, ref_support={ref_support}")

rq3_sens_df = pd.DataFrame(rq3_sensitivity)
rq3_sens_df


### 8.1 Top patterns and novel patterns at each sensitivity setting


In [ ]:
# Show top patterns and check for novel ones at each min_support level.
for minsup_frac in PREFIXSPAN_MINSUP_FRACS:
    df = rq3_patterns[minsup_frac]
    print(f"\n================ PrefixSpan minsup={minsup_frac} ================")
    if len(df) == 0:
        print("  (no patterns)")
        continue

    print(f"--- Top 10 patterns overall (length-2 will dominate by support) ---")
    display(df.sort_values("seq_support", ascending=False).head(10)[
        ["seq_support_frac", "length", "pattern_titles", "unordered_counterpart_frequent"]
    ].reset_index(drop=True))

    long_df = df[df["length"] >= 3].sort_values("seq_support", ascending=False)
    if len(long_df) == 0:
        print(f"--- Top patterns of length >= 3: NONE at minsup={minsup_frac} ---")
    else:
        print(f"--- Top 10 patterns of length >= 3 "
              f"({len(long_df):,} total of length >= 3 at this minsup) ---")
        display(long_df.head(10)[
            ["seq_support_frac", "length", "pattern_titles", "unordered_counterpart_frequent"]
        ].reset_index(drop=True))

    novel = df[df["novel_vs_unordered"]].sort_values("seq_support", ascending=False)
    if len(novel) == 0:
        print(f"--- Novel patterns (no unordered counterpart): NONE ---")
    else:
        print(f"--- Top novel patterns (no unordered counterpart) ---")
        display(novel.head(10)[
            ["seq_support_frac", "length", "pattern_titles"]
        ].reset_index(drop=True))


### 8.2 RQ3 Findings

The hypothesis that sequential mining would find structure invisible to unordered mining **is not supported**.

- **Zero novel patterns at any min_support level.** PrefixSpan found 42 patterns at minsup=0.10, 783 at minsup=0.05, and 3,587 at minsup=0.03 — 4,412 total. Every single one also exists as a frequent unordered itemset in RQ1 at the same support. The novelty rate is 0.0 at all three levels.
- **This holds even for length-3 patterns.** Length-3 patterns are where ordering has the best chance of mattering (franchise watch orders, etc.). PrefixSpan found 0 length-3 patterns at minsup=0.10, 9 at 0.05, and 374 at 0.03. All 383 have frequent unordered counterparts.
- **The patterns that appeared are exactly what we expected to be novel — and they were not.** Star Wars IV → V → VI in release order, the Lord of the Rings trilogy, Godfather I → II — these are classic "franchise order" patterns. But every one of them is also a frequent unordered triple at the same support. On MovieLens, co-consumption frequency dominates over ordering.
- **An interesting artifact in the data.** Some Lord of the Rings patterns appear in *reverse* release order (e.g., Return of the King → Fellowship → Two Towers). This happens because MovieLens timestamps record when a user *rated* a film, not when they *watched* it. Users often rate a trilogy in bulk after the final film, starting with the most recent. This means we cannot tell whether ordering truly carries no signal or whether the signal is just lost in the timestamp noise. See Limitations (§10).

**RQ3 is disconfirmed.** Across three min_support levels and 4,412 patterns (including all 383 of length ≥ 3), sequential mining found nothing that unordered mining had not already captured. We did not adjust thresholds after seeing the results — the sweep was fixed before running.


## 9. Synthesis Across RQs

Two hypotheses were confirmed and one was disconfirmed. Together, they tell a coherent story about how pattern mining works on MovieLens 25M.

1. **Support threshold is a modeling choice, not just a parameter (RQ1).** Itemset counts change by three orders of magnitude (563 to 410,690) as support drops from 0.10 to 0.015, and the *content* shifts systematically: blockbusters at high supports, genre cohorts in the middle, and niche long-tail patterns at the bottom.

2. **Confidence and lift give completely different rule rankings (RQ2).** Jaccard overlap = 0 at K=10, 25, 50, and 100. Confidence favors rules with popular consequents (Godfather); lift favors franchise pairs (Harry Potter, Bourne). A system using only one metric would never see the other's top rules.

3. **Sequential ordering adds no new information here (RQ3).** All 4,412 sequential patterns, including all 383 of length >= 3, have unordered counterparts. Even the franchise-order patterns we expected to be novel (Star Wars, LotR, Godfather) are also frequent unordered triples. This is bounded by a dataset limitation (timestamps represent rating events, not viewing events).

**Overall picture:** MovieLens 25M's positive-rating structure is dominated by co-consumption frequency. Popularity drives the confidence ranking (RQ2) and the top of both ordered (RQ3) and unordered (RQ1) pattern lists. Finer structure like franchise affinity only appears when we correct for popularity (lift) or drop below blockbuster thresholds (band view). Adding ordering as a discriminator (RQ3) does not surface anything new.


## 9.5 Export: Run Summary

The cell below writes all key results into a single Markdown file for easy sharing.


In [ ]:
import io

buf = io.StringIO()
def w(*a):  buf.write(" ".join(str(x) for x in a) + "\n")
def wtbl(title, df):
    w(f"### {title}")
    w()
    w("```")
    w(df.to_string())
    w("```")
    w()

w("# CSCE 676 — Semester Project run summary")
w(f"_generated from notebook run_")
w()

w("## Data shapes (post-pruning)")
w(f"- positive-rating rows after pruning to top {TOP_K_ITEMS} items: {len(pruned):,}")
w(f"- unique users in pruned data: {pruned['userId'].nunique():,}")
w(f"- unique items in pruned data: {pruned['movieId'].nunique():,}")
w(f"- basket_df shape: {basket_df.shape}")
w(f"- basket density: {basket_df.values.mean()*100:.4f}%")
w()

wtbl("RQ1: sweep table", rq1_df)
wtbl("RQ1: itemset-length breakdown by support", size_df)

w("### RQ1: top singletons/pairs at support=0.10 (blockbuster top)")
w()
for L in [1, 2]:
    df = itemsets_by_support[0.10]
    df = df[df["length"] == L].sort_values("support", ascending=False).head(10).copy()
    df["titles"] = df["itemsets"].apply(lambda x: titles(list(x)))
    wtbl(f"Top 10 len-{L} itemsets @ support=0.10", df[["support","length","titles"]].reset_index(drop=True))

w("### RQ1: items unique to each support band")
w()
bands = [(0.07, 0.10), (0.05, 0.07), (0.03, 0.05), (0.02, 0.03), (0.015, 0.02)]
for lo, hi in bands:
    for L in [2, 3]:
        if lo not in itemsets_by_support:
            continue
        df = itemsets_by_support[lo]
        df = df[(df["length"] == L) & (df["support"] >= lo) & (df["support"] < hi)]               .sort_values("support", ascending=False).head(10).copy()
        if len(df) == 0:
            continue
        df["titles"] = df["itemsets"].apply(lambda x: titles(list(x)))
        wtbl(f"Top 10 len-{L} itemsets in band [{lo}, {hi})",
             df[["support","length","titles"]].reset_index(drop=True))

w("## RQ2: rules")
w(f"- total rules at support={RULE_SUPPORT}, min_confidence={MIN_CONFIDENCE}: {len(rules):,}")
w()
wtbl("Top 10 rules by CONFIDENCE",
     rules.sort_values("confidence", ascending=False).head(10)[
         ["antecedent_titles","consequent_titles","support","confidence","lift"]
     ].reset_index(drop=True))
wtbl("Top 10 rules by LIFT",
     rules.sort_values("lift", ascending=False).head(10)[
         ["antecedent_titles","consequent_titles","support","confidence","lift"]
     ].reset_index(drop=True))

w("### RQ2: Jaccard overlap between confidence-ranked and lift-ranked top-K")
w()
w("```")
for K in [10, 25, 50, 100]:
    top_c = set(rules.nlargest(K, "confidence").apply(
        lambda r: (tuple(sorted(r["antecedent_titles"])), tuple(sorted(r["consequent_titles"]))), axis=1))
    top_l = set(rules.nlargest(K, "lift").apply(
        lambda r: (tuple(sorted(r["antecedent_titles"])), tuple(sorted(r["consequent_titles"]))), axis=1))
    inter = len(top_c & top_l)
    union = len(top_c | top_l)
    jac = inter / union if union else 0.0
    w(f"Top-{K:>3}: overlap={inter}, Jaccard={jac:.3f}")
w("```")
w()

w("## RQ3: sensitivity sweep")
wtbl("RQ3: per-minsup sensitivity", rq3_sens_df)

w("_Note: the Top-15 overall tables below are identical across min_support "
  "settings by construction — sorting by raw support always surfaces the "
  "same highest-support length-2 pairs. The informative comparison is the "
  "sensitivity table above plus the length >= 3 tables below, which show "
  "patterns that only surface as the min_support threshold drops._")
w()

for minsup_frac in PREFIXSPAN_MINSUP_FRACS:
    df = rq3_patterns[minsup_frac]
    if len(df) == 0:
        continue
    wtbl(f"Top 15 sequential patterns @ PrefixSpan minsup={minsup_frac}",
         df.sort_values("seq_support", ascending=False).head(15)[
             ["seq_support_frac","length","pattern_titles","unordered_counterpart_frequent"]
         ].reset_index(drop=True))

    long_df = df[df["length"] >= 3].sort_values("seq_support", ascending=False)
    if len(long_df):
        wtbl(f"Top 15 length>=3 patterns @ PrefixSpan minsup={minsup_frac} "
             f"({len(long_df):,} total of length >= 3)",
             long_df.head(15)[
                 ["seq_support_frac","length","pattern_titles","unordered_counterpart_frequent"]
             ].reset_index(drop=True))
    else:
        w(f"### Length>=3 patterns @ PrefixSpan minsup={minsup_frac}")
        w("_None. No length-3 or longer sequential patterns at this min_support._")
        w()

    novel = df[df["novel_vs_unordered"]].sort_values("seq_support", ascending=False)
    if len(novel):
        wtbl(f"Top 15 NOVEL patterns @ PrefixSpan minsup={minsup_frac}",
             novel.head(15)[["seq_support_frac","length","pattern_titles"]].reset_index(drop=True))
    else:
        w(f"### Novel patterns @ PrefixSpan minsup={minsup_frac}")
        w("_None. Every sequential pattern has an unordered counterpart frequent at the reference support._")
        w()

OUT = "semester_project_run_summary.md"
with open(OUT, "w", encoding="utf-8") as f:
    f.write(buf.getvalue())

print(f"Wrote {OUT} ({len(buf.getvalue()):,} chars)")


## 10. Limitations

- **Sampling.** We used 25,000 users for the basket matrix and 6,000 for the sequential analysis, sampled from the full population. A full-data run is possible on a 32 GB machine, but the sample is large enough that support estimates and pattern counts are stable.

- **Positive-rating threshold.** We define "positive" as rating >= 4.0 (from Checkpoint 2). A different threshold would change basket density and absolute support values, but the structural conclusions should hold since they are driven by co-consumption frequency.

- **Structural evaluation only.** We evaluate patterns using mining metrics (support, confidence, lift, novelty), not downstream recommendation performance. A train/test split with metrics like MAP or NDCG would be a natural extension.

- **max_length=3 in RQ1.** Itemset length is capped at 3 for all support levels. Length-4 patterns likely exist at low supports but were not mined due to runtime constraints. The length-2/3 sweep still demonstrates the growth rate clearly.

- **Timestamps are rating events, not viewing events.** This is the biggest caveat for RQ3. MovieLens records when users *rated* films, not when they *watched* them. Some LotR patterns appear in reverse release order because users often rate trilogies in bulk starting with the newest film. We cannot tell whether ordering truly carries no signal or whether the signal is lost in the timestamp noise. A dataset with actual viewing timestamps (e.g., streaming logs) might show different results.

- **Algorithm note.** FP-Growth was the primary miner because mlxtend's Apriori runs out of memory at low supports on this matrix size. Apriori with `low_memory=True` was used as a cross-check at supports 0.10 and 0.07, where both algorithms agreed exactly.


## 11. Conclusions

This notebook answered three research questions from Checkpoint 2 on the full MovieLens 25M dataset using a 25,000-user basket matrix (top 2,500 items) and a 6,000-user sequential sample.

**RQ1 — Confirmed.** Itemset counts grow from 563 at support=0.10 to 410,690 at support=0.015, mostly from length-3 combinations. The band view shows the content changes at each level: blockbusters at strict supports, franchise cohorts in the middle, and niche long-tail patterns at permissive supports. Support is a modeling decision, not just a tuning parameter.

**RQ2 — Confirmed, strongly.** From 14,442 rules (support=0.05, min_confidence=0.5), the top-K-by-confidence and top-K-by-lift rankings share **zero rules at K=10, 25, 50, and 100**. Confidence-top rules all point at popular movies (Godfather, confidence=0.97); lift-top rules are all franchise pairs (Harry Potter, Bourne, lift=9.29). The two metrics are completely orthogonal.

**RQ3 — Disconfirmed.** PrefixSpan found 4,412 sequential patterns across three min_support levels (42, 783, and 3,587 respectively), including 383 of length >= 3. Every single one has a frequent unordered counterpart. The franchise-order patterns we expected to be novel (Star Wars, LotR, Godfather) are all present but also exist as unordered triples. Ordering adds no new information on this data. One caveat: MovieLens timestamps reflect rating order, not viewing order, so we cannot rule out that a dataset with true viewing timestamps might show different results.

**Overall:** MovieLens 25M is dominated by co-consumption frequency. Popularity drives confidence rankings (RQ2) and the top of both ordered (RQ3) and unordered (RQ1) pattern lists. Finer structure only appears when correcting for popularity (lift) or lowering the support threshold (band view).

**Note:** The RQ3 sweep points (0.10, 0.05, 0.03) were fixed before running. We did not adjust thresholds after seeing results. Reporting the null result honestly is the right approach.


## 12. Collaboration Declaration

### Collaborators
N/A

### Web Sources
- Apriori (mlxtend): https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/
- FP-Growth (mlxtend): https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/fpgrowth/
- Association Rules (mlxtend): https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/
- PrefixSpan (Python): https://github.com/chuanconggao/PrefixSpan-py
- PrefixSpan (SPMF reference): https://www.philippe-fournier-viger.com/spmf/PrefixSpan.php
- MovieLens Dataset: https://grouplens.org/datasets/movielens/

### AI Tools

ChatGPT / Claude:
- Helped shape the RQ1/RQ2/RQ3 execution design into a single coherent story.
- Helped explain mlxtend's association_rules output columns and PrefixSpan's topk / closed / frequent semantics.
- Helped structure the novelty comparison between sequential patterns and unordered itemsets, including the multi-support sensitivity sweep used in this local build.

### References

[1] "FP-Growth Algorithm," Wikipedia. https://en.wikipedia.org/wiki/FP-growth_algorithm

[2] J. Han, H. Cheng, D. Xin, and X. Yan, "Frequent pattern mining: current status and future directions," 2007. https://www.cs.uic.edu/~liub/PMarchive/Survey-SPM.pdf

[3] J. Pei et al., "PrefixSpan: Mining Sequential Patterns Efficiently by Prefix-Projected Pattern Growth," ICDE 2001.

[4] Järv, P. (2019). *Predictability limits in session-based next item recommendation.* RecSys '19, 146–150. https://doi.org/10.1145/3298689.3346990

[5] Koren, Y. (2008). *Factorization meets the neighborhood: A multifaceted collaborative filtering model.* KDD '08, 426–434. https://doi.org/10.1145/1401890.1401944
